# 29. 커뮤니티 제품 은어 별칭표

| | |
|---|---|
| 만드는 것 | `data/scent_knowledge/community_product_alias_v1.csv` |
| 출처 | `masiondior.github.io/perfume_finder/mapping.json` (사용자 제공, 2026-09-11 수집) |
| API | **호출하지 않는다** |
| 작성 | 2026-09-11 |

## 왜 Domain Lexicon 에 넣지 않는가

`domain_lexicon_v1.csv` 는 **표현 → accord / note / 스키마 필드** 를 매핑한다.
이 28개는 **은어 → 향수 제품** 이므로 종류가 다르다.

```
사전            포근한  →  powdery (ACCORD)
이 파일         로체    →  로스트 체리  →  Tom Ford Lost Cherry
```

넣을 수 없는 이유가 셋이다.

1. `spec.md` §4.3 의 `candidate_type` 은 `ACCORD` / `NOTE` / `FIELD` 뿐이다. `PERFUME` 이 없다
2. 팀 문서가 범위 밖으로 선언했다 — *"제품 유사도 데이터 … **어휘 사전의 범위 밖이고,
   이 문서로는 해결되지 않는다**"*
3. `spec.md` §3 의 Stage 1 출력에 제품명을 담을 필드가 없다.
   즉 **지금 파이프라인에 꽂을 자리가 없다**

`25_stage1_scoring_alias_v1.csv` 가 채점기용 별칭표로 따로 있는 것과 같은 구조로 분리한다.

## 그래서 이 파일은 언제 쓰는가

팀 문서 20-C장이 분류한 질문 4모드 중 **참조형**(써 본 향수를 기준점으로 삼는 질문)을
처리하게 될 때다. `spec.md` §8 에 그 작업이 아직 없으므로, 지금은 **수집해 두는 것**이 목적이다.

## 0. 실행 조건과 한계

**외부 데이터 출처다.** `AGENTS.md` 가 *"Do not introduce a new external data source without
user approval"* 과 *"record provenance when the result will become reusable knowledge"* 를
요구한다. 사용자가 사이트를 제시하고 은어 수집을 지시했으며(2026-09-11), 출처 URL 과 수집일을
파일에 기록한다.

**원본 데이터의 결함을 고쳐서 넣는다.** `mapping.json` 에 한 문자열 안에 은어 두 개가
쉼표로 들어간 항목이 둘 있다 — `"봐달장, 달짱"`, `"박아라, 바카라"`. 분리해서 넣고
`match_note` 에 기록한다.

**한계**

- **영어 제품명 대응은 내가 판단한 것이다.** 원본에는 한국어 정식명만 있다.
  `perfumes.csv` 조회로 확인했으나 동명이곡·연도 변형이 있는 경우 `AMBIGUOUS` 로 둔다
- **은어 28개는 이 사이트 운영자가 정리한 목록이며 빈도 근거가 없다.**
  어느 은어가 실제로 얼마나 쓰이는지는 측정하지 않았다
- **`로체` 같은 은어가 들어왔을 때 무엇을 할지는 정해지지 않았다.**
  제품을 찾은 뒤 유사 향수를 주는 기능이 아직 없다
- 이 파일은 **사전이 아니다.** accord 매핑을 담지 않는다

**하지 않는 것** — `domain_lexicon_v1.csv` 수정, `spec.md` 스키마 변경, DC 본문 크롤링,
평가 데이터 수정.

In [1]:
import hashlib
import json
import pathlib

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 46)
pd.set_option("display.width", 240)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

SOURCE_URL = "https://masiondior.github.io/perfume_finder/mapping.json"
COLLECTED_AT = "2026-09-11"
print("REPORT_ONLY:", REPORT_ONLY)

REPORT_ONLY: False


## 1. 경로 · 입력 해싱 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
KNOWLEDGE_DIR = PROJECT_ROOT / "data" / "scent_knowledge"
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATHS = {"perfumes": PROJECT_ROOT / "perfumes.csv"}
OUTPUT_PATHS = {"alias": KNOWLEDGE_DIR / "community_product_alias_v1.csv"}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

PROTECTED = {
    (KNOWLEDGE_DIR / "domain_lexicon_v1.csv").resolve(),
    (OUTPUT_DIR / "25_stage1_scoring_alias_v1.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "semantic_bridge"
     / "22_pilot_human_evaluation.csv").resolve(),
}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 의 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
perfumes,cec1ea0b49885303


## 2. 원본 데이터 — 수집 시점의 내용을 그대로 박아둔다

원본은 외부 URL 이라 언제든 바뀔 수 있다. 재현을 위해 **수집 당시의 28개 항목을 노트북에
그대로 적어둔다.** 원본 형식은 `{정식 한국어명: [은어, …]}` 이다.

In [3]:
# masiondior.github.io/perfume_finder/mapping.json, 2026-09-11 수집분 원문
RAW_MAPPING = {
    "제라늄 뿌르 무슈": ["제뿌무"],
    "문라이트 인 헤븐": ["문인헤"],
    "휘그 인 퓨전": ["휘강보종"],
    "브와 임페리얼": ["부왕패렬"],
    "누보몽드": ["누보지드"],
    "베티베디베": ["야티야디야"],
    "망고 타이 라임": ["망타라"],
    "화이트 자고라": ["화자"],
    "도쿄 블룸": ["도쿄불륜"],
    "애프터미드나잇": ["애미나이"],
    "서브라임 발키스": ["쉬블럼발뽀뽀", "발뽀뽀"],
    "디옴 옴므 인텐스": ["디옴인"],
    "디올 옴므 코롱": ["디옴코"],
    "발라드 소바쥬": ["발바쥬", "밟아줘"],
    "디올리비에라": ["디비자라"],
    "브아 다르장": ["봐달장, 달짱"],          # 원본 결함: 한 문자열에 은어 2개
    "바카라 루쥬 540": ["박아라, 바카라"],     # 원본 결함: 한 문자열에 은어 2개
    "아쿠아 셀레스티아 포르테": ["아셀포"],
    "아쿠아 셀레스티아 코롱 포르테": ["아셀코포"],
    "아쿠아 유니버셜 코롱 포르테": ["아유코포"],
    "아쿠아 미디어 코롱 포르테": ["아미코포"],
    "롬므 아라로즈": ["롬라로즈"],
    "로스트 체리": ["로체"],
    "섹스 온 더 시 네롤리": ["섹섹네"],
    "마태우 NO5": ["마넘파"],
    "망트 라 졸리": ["파란포션", "파포"],
    "베이비캣": ["아가단또", "베캣", "단또"],
    "엔젤스 셰어": ["엔젤스 쉐어", "엔쉐", "엔셰"],
}
print(f"원본 항목 {len(RAW_MAPPING)}개 / 은어 총 "
      f"{sum(len(v) for v in RAW_MAPPING.values())}개 (분리 전)")

원본 항목 28개 / 은어 총 35개 (분리 전)


## 3. 영어 제품명 대응 — 내가 판단한 부분

원본에는 한국어 정식명만 있다. `perfumes.csv` 에서 찾은 Fragrantica id 를 적는다.
**이 대응은 내 판단이므로 검증 셀에서 id 가 실제로 존재하는지 확인한다.**

찾지 못했거나 후보가 갈리는 것은 id 를 비우고 사유를 적는다.

In [4]:
# 정식 한국어명 → (Fragrantica id, 판정, 주석)
RESOLUTION = {
    "제라늄 뿌르 무슈": ("5911", "CONFIRMED", ""),
    "문라이트 인 헤븐": ("35973", "CONFIRMED", ""),
    "휘그 인 퓨전": ("73092", "CONFIRMED", "Essential Parfums Fig Infusion. 은어 '휘강보종'의 유래는 미상"),
    "브와 임페리얼": ("64338", "CONFIRMED", "Extrait 등 파생 제품이 따로 있다"),
    "누보몽드": ("49753", "CONFIRMED", ""),
    "베티베디베": ("", "AMBIGUOUS", "Vetyver 계열 후보가 여럿이라 특정 못 함"),
    "망고 타이 라임": ("16436", "CONFIRMED", ""),
    "화이트 자고라": ("18596", "CONFIRMED", ""),
    "도쿄 블룸": ("14838", "CONFIRMED", ""),
    "애프터미드나잇": ("14840", "CONFIRMED", ""),
    "서브라임 발키스": ("4223", "CONFIRMED", ""),
    "디옴 옴므 인텐스": ("1771", "CONFIRMED", "Dior Homme Intense 2007. 연도 변형이 여럿 있다"),
    "디올 옴므 코롱": ("1770", "CONFIRMED", ""),
    "발라드 소바쥬": ("", "NOT_FOUND", "Dior Ballade Sauvage 로 추정되나 코퍼스에 없다"),
    "디올리비에라": ("81847", "CONFIRMED", ""),
    "브아 다르장": ("1377", "CONFIRMED", ""),
    "바카라 루쥬 540": ("33519", "CONFIRMED", "MFK 본품. Extrait·Hair Mist 등 파생이 5종 더 있다"),
    "아쿠아 셀레스티아 포르테": ("49217", "CONFIRMED", ""),
    "아쿠아 셀레스티아 코롱 포르테": ("66993", "CONFIRMED", ""),
    "아쿠아 유니버셜 코롱 포르테": ("66996", "CONFIRMED", ""),
    "아쿠아 미디어 코롱 포르테": ("82030", "CONFIRMED", ""),
    "롬므 아라로즈": ("", "NOT_FOUND", "Frederic Malle L'Homme a la Rose 로 추정되나 코퍼스에 없다"),
    "로스트 체리": ("51411", "CONFIRMED", ""),
    "섹스 온 더 시 네롤리": ("", "AMBIGUOUS", "Demeter Sex on the Beach 계열 후보가 있으나 '네롤리'와 안 맞는다"),
    "마태우 NO5": ("", "AMBIGUOUS", "은어 '마넘파'는 '마 넘버 파이브'로 읽히나 브랜드를 특정 못 함"),
    "망트 라 졸리": ("", "AMBIGUOUS", "은어 '파란포션'이 색을 가리키나 제품을 특정 못 함"),
    "베이비캣": ("63448", "CONFIRMED", "YSL Black Opium Baby Cat Collector"),
    "엔젤스 셰어": ("62615", "CONFIRMED", "By Kilian. 파생 제품이 3종 더 있다"),
}
assert set(RESOLUTION) == set(RAW_MAPPING), "원본과 대응표의 항목이 다릅니다"
print(f"대응 적음 {len(RESOLUTION)}개")

대응 적음 28개


## 4. 조립 — 원본 결함을 고치고 id 를 검증한다

원본에서 한 문자열에 쉼표로 은어 두 개가 들어간 항목을 분리한다.
그리고 적어둔 Fragrantica id 가 실제로 존재하는지 `perfumes.csv` 에서 확인한다.

In [5]:
perfumes = pd.read_csv(INPUT_PATHS["perfumes"], usecols=["id", "name", "brand"],
                       dtype={"id": str})
BY_ID = {r.id: (r.brand, r["name"]) for _, r in perfumes.iterrows()}

rows = []
split_fixed = []
for ko, slangs in RAW_MAPPING.items():
    # 원본 결함 보정: 한 문자열 안의 쉼표를 분리한다
    clean = []
    for s in slangs:
        parts = [x.strip() for x in s.split(",") if x.strip()]
        if len(parts) > 1:
            split_fixed.append((ko, s, parts))
        clean += parts

    pid, status, note = RESOLUTION[ko]
    if pid and pid not in BY_ID:
        raise RuntimeError(f"적어둔 id 가 코퍼스에 없습니다: {ko} → {pid}")
    brand, name = BY_ID.get(pid, ("", ""))

    rows.append({
        "entry_id": f"kr.slang.{len(rows) + 1:02d}",
        "slang": "|".join(clean),
        "canonical_ko": ko,
        "perfume_id": pid,
        "brand_en": brand,
        "name_en": name,
        "match_status": status,
        "match_note": note,
        "source_url": SOURCE_URL,
        "collected_at": COLLECTED_AT,
    })

alias_df = pd.DataFrame(rows)

print(f"원본 결함 보정 {len(split_fixed)}건")
for ko, orig, parts in split_fixed:
    print(f"   {ko}: '{orig}' → {parts}")
print(f"\n행 {len(alias_df)} / 은어 총 {sum(len(r.split('|')) for r in alias_df.slang)}개")
print(alias_df.match_status.value_counts().to_dict())
display(alias_df[["slang", "canonical_ko", "perfume_id", "brand_en", "name_en",
                  "match_status"]])

원본 결함 보정 2건
   브아 다르장: '봐달장, 달짱' → ['봐달장', '달짱']
   바카라 루쥬 540: '박아라, 바카라' → ['박아라', '바카라']

행 28 / 은어 총 37개
{'CONFIRMED': 22, 'AMBIGUOUS': 4, 'NOT_FOUND': 2}


,slang,canonical_ko,perfume_id,brand_en,name_en,match_status
0,제뿌무,제라늄 뿌르 무슈,5911,Frederic Malle,Geranium Pour Monsieur,CONFIRMED
1,문인헤,문라이트 인 헤븐,35973,By Kilian,Moonlight in Heaven,CONFIRMED
2,휘강보종,휘그 인 퓨전,73092,Essential Parfums,Fig Infusion,CONFIRMED
3,부왕패렬,브와 임페리얼,64338,Essential Parfums,Bois Impérial,CONFIRMED
4,누보지드,누보몽드,49753,Louis Vuitton,Nouveau Monde,CONFIRMED
5,야티야디야,베티베디베,,,,AMBIGUOUS
6,망타라,망고 타이 라임,16436,Jo Loves,Mango Thai Lime,CONFIRMED
7,화자,화이트 자고라,18596,The Different Company,White Zagora,CONFIRMED
8,도쿄불륜,도쿄 블룸,14838,The Different Company,Tokyo Bloom,CONFIRMED
9,애미나이,애프터미드나잇,14840,The Different Company,After Midnight,CONFIRMED


### 확인된 제품의 accord — 이 파일이 우리 데이터와 실제로 이어지는가

In [6]:
acc = pd.read_csv(INPUT_PATHS["perfumes"], usecols=["id", "accords"], dtype={"id": str})
ACC = dict(zip(acc.id, acc.accords.fillna("")))

ok = alias_df[alias_df.perfume_id != ""]
print(f"id 가 확인된 {len(ok)}개 제품의 상위 accord\n")
for r in ok.to_dict("records"):
    a = ACC.get(r["perfume_id"], "")
    top = " ".join(p for p in a.split("|")[:4])
    print(f"  {r['slang']:22s} {r['name_en'][:30]:32s} {top}")
missing_acc = [r["canonical_ko"] for r in ok.to_dict("records")
               if not ACC.get(r["perfume_id"])]
print(f"\naccord 가 비어 있는 제품: {len(missing_acc)}개 {missing_acc}")

id 가 확인된 22개 제품의 상위 accord

  제뿌무                    Geranium Pour Monsieur           aromatic:100 fresh spicy:96 green:70 warm spicy:54
  문인헤                    Moonlight in Heaven              sweet:100 citrus:100 tropical:83 aromatic:68
  휘강보종                   Fig Infusion                     woody:100 fruity:65 citrus:60 sweet:52
  부왕패렬                   Bois Impérial                    woody:100 fresh spicy:55 aromatic:45 oud:36
  누보지드                   Nouveau Monde                    warm spicy:100 leather:98 oud:79 cacao:63
  망타라                    Mango Thai Lime                  tropical:100 aromatic:95 citrus:90 fruity:80
  화자                     White Zagora                     citrus:100 white floral:82 floral:80 fruity:46
  도쿄불륜                   Tokyo Bloom                      yellow floral:100 fresh spicy:81 green:78 powdery:78
  애미나이                   After Midnight                   amber:100 citrus:87 powdery:73 white floral:73
  쉬블럼발뽀뽀|발뽀뽀             Sublime Balk

## 5. 저장

In [7]:
write_output(OUTPUT_PATHS["alias"],
             lambda p: alias_df.to_csv(p, index=False, encoding="utf-8-sig"))

if not REPORT_ONLY:
    reread = pd.read_csv(OUTPUT_PATHS["alias"], keep_default_na=False, dtype=str)
    if len(reread) != len(alias_df):
        raise ValueError(f"저장 결과 행 수 불일치: {len(reread)} != {len(alias_df)}")
    print(f"검증 통과 — {len(reread)}행 / 컬럼 {len(reread.columns)}개")

input_hashes_after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in input_hashes_before
           if input_hashes_before[k] != input_hashes_after[k]]
if changed:
    raise RuntimeError(f"입력이 변경됐습니다: {changed}")
print("입력 해시 동일")
for path in sorted(PROTECTED):
    print(f"보호 대상 미변경 확인: {path.name}  {'존재' if path.is_file() else '없음'}")

저장: data\scent_knowledge\community_product_alias_v1.csv
검증 통과 — 28행 / 컬럼 10개
입력 해시 동일
보호 대상 미변경 확인: 25_stage1_scoring_alias_v1.csv  존재
보호 대상 미변경 확인: domain_lexicon_v1.csv  존재
보호 대상 미변경 확인: 22_pilot_human_evaluation.csv  존재
보호 대상 미변경 확인: 13_stage1_golden_set_v1_200.xlsx  존재
